In [1]:
import pandas as pd
import numpy as np
import pycountry
from tqdm import tqdm
from pathlib import Path


`i` here is the origin country.
`j` here is the destination country.

In [2]:
# PyCountry mapping
records = []
for c in pycountry.countries:
    records.append({
        "country_name": c.name,
        "iso3":         c.alpha_3,
        "iso2":         c.alpha_2,
        "iso_numeric":  c.numeric,
    })

df_iso = pd.DataFrame(records)
print(df_iso.shape)
print(df_iso.head())

(249, 4)
    country_name iso3 iso2 iso_numeric
0          Aruba  ABW   AW         533
1    Afghanistan  AFG   AF         004
2         Angola  AGO   AO         024
3       Anguilla  AIA   AI         660
4  Åland Islands  ALA   AX         248


## PIP

In [3]:
# IMF CPIS
df_pip = pd.read_csv('../Clean/IMF_PIP.csv')
# Now we only keep the necessary columns
df_pip = df_pip[["iso3_i", "iso3_j", "year", "value"]]
# rename value to pip
df_pip = df_pip.rename(columns={"value": "pip"})

df_pip["year"] = df_pip["year"].astype(int)
df_pip["pip"] = df_pip["pip"].astype(float)

df_pip

,iso3_i,iso3_j,year,pip
0,CWX,DEU,2017,1.343000
1,CWX,GBR,2017,25.710000
2,CWX,NOR,2017,1.915000
3,CWX,BEL,2017,0.190000
4,CWX,SWE,2017,1.650000
...,...,...,...,...
89956,TX983,DNK,2024,155.119069
89957,TX983,JPN,2024,103.945040
89958,TX983,SVN,2024,516.535885
89959,TX983,MNG,2024,40.788969


## Market Cap

In [4]:
df_mkt_cap = pd.read_csv('../Clean/market_cap.csv')
df_mkt_cap = df_mkt_cap.rename({"Year": "year"}, axis=1)
# Limit to useful columns
df_mkt_cap = df_mkt_cap[["Country Code", "year", "Market Cap"]]
# Rename Country Code to iso3
df_mkt_cap = df_mkt_cap.rename({"Country Code": "iso3", "Market Cap": "market_cap"}, axis=1)

df_mkt_cap["year"] = df_mkt_cap["year"].astype(int)
df_mkt_cap["market_cap"] = df_mkt_cap["market_cap"].astype(float)

df_mkt_cap.isna().mean()

iso3          0.000000
year          0.000000
market_cap    0.789417
dtype: float64

## Bilateral cultural distance

In [5]:
# df_wvs = pd.read_csv('../Clean/WVS_Cultural_Distance.csv')
# # Rename to iso3_i and iso3_j
# df_wvs = df_wvs.rename({"country_i": "iso3_i", "country_j": "iso3_j"}, axis=1)

# df_wvs["year"] = df_wvs["year"].astype(int)
# df_wvs["cultural_distance"] = df_wvs["cultural_distance"].astype(float)

# df_wvs

In [6]:
df_hof = pd.read_csv("../Clean/hofstede_distances.csv")
df_hof

,iso3_i,iso3_j,d_cul
0,AFE,AFE,0.000000
1,AFE,AFW,47.116876
2,AFE,ALB,NaN
3,AFE,ALG,NaN
4,AFE,AND,NaN
...,...,...,...
12316,ZIM,URU,NaN
12317,ZIM,VEN,NaN
12318,ZIM,VIE,NaN
12319,ZIM,ZAM,NaN


## CEPII Gravity

In [7]:
# Gravity (Lang + Dist)
df_gravity = pd.read_csv('../Clean/cepii_gravity_language.csv')
# Join to get iso3 codes for both origin and destination
# It is iso3, though we renamed to iso in data.
df_gravity = df_gravity.merge(df_iso.add_suffix('_o'), left_on="iso3_o", right_on="iso3_o", how="left")
df_gravity = df_gravity.merge(df_iso.add_suffix('_d'), left_on="iso3_d", right_on="iso3_d", how="left")
# Select only some columns
df_gravity = df_gravity[["iso3_o", "iso3_d", "year", "distance_km", "shared_border", "common_legal_origin", "language_proximity", "language_spoken_share", "common_official_language"]]
# Remove duplicates
df_gravity = df_gravity.drop_duplicates(subset=["iso3_o", "iso3_d", "year"], keep="last")
# Rename iso3_o and iso3_d to match other datasets
df_gravity = df_gravity.rename({"iso3_o": "iso3_i", "iso3_d": "iso3_j"}, axis=1)

df_gravity["year"] = df_gravity["year"].astype(int)
df_gravity[["distance_km", "shared_border", "common_legal_origin", "language_proximity", "language_spoken_share", "common_official_language"]] \
    = df_gravity[["distance_km", "shared_border", "common_legal_origin", "language_proximity", "language_spoken_share", "common_official_language"]].astype(float)
    
# Since the gravity columns will not change a lot. We will
# group by them and get their mean value so we fill all 
# missing values.
# NOTE: We do not use distance from here anymore.
df_gravity = df_gravity.groupby(["iso3_i", "iso3_j"])[["shared_border", "common_legal_origin",
                                                           "language_proximity", "language_spoken_share", 
                                                           "common_official_language"]].mean().reset_index()

df_gravity

,iso3_i,iso3_j,shared_border,common_legal_origin,language_proximity,language_spoken_share,common_official_language
0,ABW,ABW,0.0,1.0,NaN,NaN,NaN
1,ABW,AFG,0.0,1.0,0.128432,0.0000,0.0
2,ABW,AGO,0.0,1.0,0.383448,0.0000,0.0
3,ABW,AIA,0.0,0.0,0.219024,0.3864,0.0
4,ABW,ALB,0.0,1.0,0.119646,0.0000,0.0
...,...,...,...,...,...,...,...
59044,ZWE,YMD,0.0,0.0,NaN,NaN,NaN
59045,ZWE,YUG,0.0,0.0,NaN,NaN,NaN
59046,ZWE,ZAF,1.0,1.0,0.126163,0.1218,1.0
59047,ZWE,ZMB,1.0,1.0,0.124937,0.0672,1.0


In [8]:
df_gravity.isna().mean().sort_values(ascending=False).head(10)

common_official_language    0.365916
language_spoken_share       0.365916
language_proximity          0.365916
common_legal_origin         0.011973
shared_border               0.004505
iso3_j                      0.000000
iso3_i                      0.000000
dtype: float64

## Distance

In [9]:
df_dist = pd.read_csv("../Clean/countries_distances.csv")
df_dist

,iso3_i,iso3_j,distance_km
0,AFG,AFG,0.000000
1,AFG,ALA,4436.559161
2,AFG,ALB,4047.637966
3,AFG,DZA,5881.179067
4,AFG,ASM,14114.459916
...,...,...,...
62495,ZWE,WLF,15332.596720
62496,ZWE,ESH,6792.169129
62497,ZWE,YEM,4361.029497
62498,ZWE,ZMB,555.974633


## MSCI Return

In [10]:
# MSCI Indices
df_msci = pd.read_csv('../Clean/MSCI_Country_ETFs_Yearly.csv')
# Change to get pct returns instead of price
df_msci = df_msci.sort_values(by=["iso3", "Year"])  # Ensure it's sorted before calculating returns
df_msci["Return"] = df_msci.groupby("iso3")["Price"].pct_change()
# Limit columns
df_msci = df_msci[["iso3", "Year", "Return"]]
# Rename Return to MSCI_Return for clarity
df_msci = df_msci.rename({"Return": "MSCI_Return", "Year": "year"}, axis=1)

df_msci["year"] = df_msci["year"].astype(int)
df_msci["MSCI_Return"] = df_msci["MSCI_Return"].astype(float)

df_msci

,iso3,year,MSCI_Return
1488,ARE,1996,NaN
1489,ARE,1997,NaN
1490,ARE,1998,NaN
1491,ARE,1999,NaN
1492,ARE,2000,NaN
...,...,...,...
1142,ZAF,2022,-0.051778
1143,ZAF,2023,0.015083
1144,ZAF,2024,0.071604
1145,ZAF,2025,0.752008


## BIS

In [11]:
df_bis = pd.read_csv("../Clean/BIS_Debt.csv")
# Rename to correct column names
df_bis = df_bis.rename({"value": "bis_debt"}, axis=1)

df_bis["year"] = df_bis["year"].astype(int)
df_bis["bis_debt"] = df_bis["bis_debt"].astype(float)

df_bis

,year,iso3,bis_debt
0,1946,FRA,NaN
1,1946,GRC,NaN
2,1946,GBR,NaN
3,1946,SWE,NaN
4,1946,ESP,NaN
...,...,...,...
21562,2024,AUT,638.666417
21563,2024,BRA,2671.559815
21564,2024,BGR,26.408053
21565,2024,POL,446.247129


## Penn World Table

In [12]:
# PWT
df_pwt = pd.read_csv('../Clean/pwt.csv')
df_pwt = df_pwt.rename({"countrycode": "iso3",}, axis=1)
# Calculate alphas as 1 - labor share, and rename to alpha
df_pwt["Alpha"] = 1 - df_pwt["Labour Share"]

df_pwt["year"] = df_pwt["year"].astype(int)
df_pwt["Alpha"] = df_pwt["Alpha"].astype(float)

df_pwt

,Real GDP,Capital Stock,Worker Population,Total Population,Labour Share,Depreciation Rate,TFP,Interest Rate,Investment Share,Human Capital,iso3,year,Alpha
0,368.250916,776.961548,NaN,0.058950,NaN,0.040318,NaN,NaN,0.431354,NaN,ABW,1970,NaN
1,401.719330,857.151245,NaN,0.058781,NaN,0.039650,NaN,NaN,0.418163,NaN,ABW,1971,NaN
2,438.229523,944.586182,NaN,0.058895,NaN,0.039095,NaN,NaN,0.419161,NaN,ABW,1972,NaN
3,478.057892,1039.993408,NaN,0.059435,NaN,0.038641,NaN,NaN,0.403070,NaN,ABW,1973,NaN
4,521.506104,1143.999634,NaN,0.060122,NaN,0.038273,NaN,NaN,0.390737,NaN,ABW,1974,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11196,44445.031250,107682.148438,5.272343,15.271368,0.533381,0.059923,1.057740,0.275977,0.130147,2.713408,ZWE,2019,0.466619
11197,40970.785156,109248.289062,5.206007,15.526888,0.533381,0.060208,0.973187,0.263760,0.119450,2.746586,ZWE,2020,0.466619
11198,44440.199219,110979.773438,5.298346,15.797210,0.533381,0.060136,1.000000,0.205160,0.153113,2.770661,ZWE,2021,0.466619
11199,47147.550781,114134.882812,5.344455,16.069056,0.533381,0.059788,0.997026,0.263871,0.172489,2.795292,ZWE,2022,0.466619


## Euro

In [13]:
# Make a dataframe given a mapping of when countries joined
# the euro currency
euro_map = {
    "AUT": 1999,
    "BEL": 1999,
    "CYP": 2008,
    "EST": 2011,
    "FIN": 1999,
    "FRA": 1999,
    "DEU": 1999,
    "GRC": 2001,
    "IRL": 1999,
    "ITA": 1999,
    "LVA": 2014,
    "LTU": 2015,
    "LUX": 1999,
    "MLT": 2008,
    "NLD": 1999,
    "PRT": 1999,
    "SVK": 2009,
    "SVN": 2007,
    "ESP": 1999
}

# Now make a date range from 1970 to 2026
dt_range = list(range(1970, 2027))
df_euro = pd.DataFrame([(iso, year) for iso in euro_map.keys() for year in dt_range], columns=["iso3", "year"])
df_euro["euro"] = df_euro.apply(lambda row: 1 if row["year"] >= euro_map[row["iso3"]] else 0, axis=1)
df_euro

,iso3,year,euro
0,AUT,1970,0
1,AUT,1971,0
2,AUT,1972,0
3,AUT,1973,0
4,AUT,1974,0
...,...,...,...
1078,ESP,2022,1
1079,ESP,2023,1
1080,ESP,2024,1
1081,ESP,2025,1


## Full dataframe

In [14]:
# ### FULL ###
# # Get min date for each country in the gravity data
# # # Exclude the df_euro, since we imposed min year
# min_year = max(df_pip["year"].min()
#                , df_mkt_cap["year"].min()
#                , df_msci["year"].min()
#                , df_pwt["year"].min()
# )
# min_year

In [15]:
# # Limit all datasets to this min year
# df_pip = df_pip[df_pip["year"] >= min_year]
# df_msci = df_msci[df_msci["year"] >= min_year]
# df_mkt_cap = df_mkt_cap[df_mkt_cap["year"] >= min_year]
# df_pwt = df_pwt[df_pwt["year"] >= min_year]
# df_euro = df_euro[df_euro["year"] >= min_year]
# df_bis = df_bis[df_bis["year"] >= min_year]

In [16]:
# Make cartesian index of year, iso3_i and iso3_j
from itertools import product
iso3_i = df_pip["iso3_i"].unique()
iso3_j = df_pip["iso3_j"].unique()
year = df_pip["year"].unique()
cartesian_index = pd.DataFrame(list(product(iso3_i, iso3_j, year)), columns=["iso3_i", "iso3_j", "year"])

df_full = pd.DataFrame(list(product(iso3_i, iso3_j, year)), columns=["iso3_i", "iso3_j", "year"])

# Join to one dataframe
df_full = df_full.merge(df_pip, left_on=["iso3_i", "iso3_j", "year"], right_on=["iso3_i", "iso3_j", "year"], how="left")
print("value" in df_full.columns)

df_full = df_full.merge(df_mkt_cap.add_suffix("_i"), left_on=["iso3_i", "year"], right_on=["iso3_i", "year_i"], how="left")
df_full = df_full.merge(df_mkt_cap.add_suffix("_j"), left_on=["iso3_j", "year"], right_on=["iso3_j", "year_j"], how="left")
df_full = df_full.drop(columns=["year_i", "year_j"])
print("value" in df_full.columns)

df_full = df_full.merge(df_hof, left_on=["iso3_i", "iso3_j"], right_on=["iso3_i", "iso3_j"], how="left")
print("value" in df_full.columns)

df_full = df_full.merge(df_dist, left_on=["iso3_i", "iso3_j"], right_on=["iso3_i", "iso3_j"], how="left")
print("value" in df_full.columns)

df_full = df_full.merge(df_gravity, left_on=["iso3_i", "iso3_j"], right_on=["iso3_i", "iso3_j"], how="left")
print("value" in df_full.columns)

df_full = df_full.merge(df_msci.add_suffix("_i"), left_on=["iso3_i", "year"], right_on=["iso3_i", "year_i"], how="left")
df_full = df_full.merge(df_msci.add_suffix("_j"), left_on=["iso3_j", "year"], right_on=["iso3_j", "year_j"], how="left")
df_full = df_full.drop(columns=["year_i", "year_j"])
print("value" in df_full.columns)

df_full = df_full.merge(df_pwt.add_suffix("_i"), left_on=["iso3_i", "year"], right_on=["iso3_i", "year_i"], how="left")
df_full = df_full.merge(df_pwt.add_suffix("_j"), left_on=["iso3_j", "year"], right_on=["iso3_j", "year_j"], how="left")
df_full = df_full.drop(columns=["year_i", "year_j"])
print("value" in df_full.columns)

df_full = df_full.merge(df_euro.add_suffix("_i"), left_on=["iso3_i", "year"], right_on=["iso3_i", "year_i"], how="left")
df_full = df_full.merge(df_euro.add_suffix("_j"), left_on=["iso3_j", "year"], right_on=["iso3_j", "year_j"], how="left")
df_full = df_full.drop(columns=["year_i", "year_j"])
print("value" in df_full.columns)

# Join BIS for each country, only i
df_full = df_full.merge(df_bis.add_suffix("_i"), left_on=["iso3_i", "year"], right_on=["iso3_i", "year_i"], how="left")
df_full = df_full.drop(columns=["year_i"])
print("value" in df_full.columns)

df_full

False
False
False
False
False
False
False
False
False


,iso3_i,iso3_j,year,pip,market_cap_i,market_cap_j,d_cul,distance_km,shared_border,common_legal_origin,...,Labour Share_j,Depreciation Rate_j,TFP_j,Interest Rate_j,Investment Share_j,Human Capital_j,Alpha_j,euro_i,euro_j,bis_debt_i
0,CWX,DEU,2017,1.34300,NaN,2.262223e+12,NaN,NaN,NaN,NaN,...,0.634760,0.036102,0.998753,0.056545,0.224437,3.670468,0.365240,NaN,1.0,NaN
1,CWX,DEU,2018,2.35200,NaN,1.755173e+12,NaN,NaN,NaN,NaN,...,0.644704,0.036375,0.997346,0.052225,0.237154,3.672922,0.355296,NaN,1.0,NaN
2,CWX,DEU,2019,0.37100,NaN,2.098174e+12,NaN,NaN,NaN,NaN,...,0.655868,0.036638,0.998371,0.048157,0.229865,3.675378,0.344132,NaN,1.0,NaN
3,CWX,DEU,2020,1.00904,NaN,2.284109e+12,NaN,NaN,NaN,NaN,...,0.657273,0.036727,0.985335,0.043165,0.232477,3.677836,0.342727,NaN,1.0,NaN
4,CWX,DEU,2021,0.52612,NaN,2.503046e+12,NaN,NaN,NaN,NaN,...,0.637342,0.036680,1.000000,0.048045,0.239041,3.686913,0.362658,NaN,1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
942667,ESH,ATF,2020,NaN,NaN,NaN,NaN,11507.09998,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
942668,ESH,ATF,2021,NaN,NaN,NaN,NaN,11507.09998,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
942669,ESH,ATF,2022,NaN,NaN,NaN,NaN,11507.09998,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
942670,ESH,ATF,2023,NaN,NaN,NaN,NaN,11507.09998,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# Compute the linguistic distance as inverse of language proximity
df_full["language_distance"] = 1 - df_full["language_proximity"]

In [18]:
# Fix df_full to only be countries that we have defined it to be
# ── Country set: EU-27 + 4 outside options ────────────────────────────────
EU27 = [
    'AUT','BEL','BGR','HRV','CYP','CZE','DNK','EST','FIN','FRA','DEU','GRC',
    'HUN','IRL','ITA','LVA','LTU','LUX','MLT','NLD','POL','PRT','ROU','SVK',
    'SVN','ESP','SWE'
]
OUTSIDE = ['USA','GBR','CHE','NOR']
COUNTRIES = EU27 + OUTSIDE          # n = 31

In [19]:
# Fill nan euro_i and euro_j with 0, since if they are not in the dataset, they are not in the euro
df_full["euro_i"] = df_full["euro_i"].fillna(0)
df_full["euro_j"] = df_full["euro_j"].fillna(0)

In [20]:
# # Manually impute home bias for the countries
# # we define a map as a percentage of the foreign pip 
# # that is invested at home. For all EU27 + 4
# home_bias_map = {
#     'AUT': 0.65,
#     'BEL': 0.58,
#     'BGR': 0.72,
#     'HRV': 0.68,
#     'CYP': 0.70,
#     'CZE': 0.64,
#     'DNK': 0.62,
#     'EST': 0.75,
#     'FIN': 0.61,
#     'FRA': 0.72,
#     'DEU': 0.69,
#     'GRC': 0.74,
#     'HUN': 0.66,
#     'IRL': 0.55,
#     'ITA': 0.71,
#     'LVA': 0.76,
#     'LTU': 0.77,
#     'LUX': 0.45,
#     'MLT': 0.60,
#     'NLD': 0.56,
#     'POL': 0.70,
#     'PRT': 0.67,
#     'ROU': 0.73,
#     'SVK': 0.65,
#     'SVN': 0.68,
#     'ESP': 0.68,
#     'SWE': 0.63,
#     'USA': 0.78,
#     'GBR': 0.67,
#     'NOR': 0.72,
#     'CHE': 0.64,
# }

# # Make a loop that pivots each year and gets the sum of
# # pip for country i per year.
# grp_iso3i_year = df_full.groupby(["iso3_i", "year"])["pip"].sum().reset_index()

# # Apply the mapping to this grouped dataframe, and calculate the home pip.
# # Which means that domestic pip must be sum(i≠j) / (1-home_bias) to get 
# # the home pip.
# grp_iso3i_year["home_pip"] = grp_iso3i_year.apply(lambda row: row["pip"] * home_bias_map.get(row["iso3_i"], 0.5) / (1 - home_bias_map.get(row["iso3_i"], 0.5)), axis=1)

# # Join the home pip as pip when i=j for that year
# for _, row in tqdm(grp_iso3i_year.iterrows(), total=len(grp_iso3i_year)):
#     iso3_i = row["iso3_i"]
#     year = row["year"]
#     home_pip = row["home_pip"]
    
#     # Update the pip in the full dataframe where iso3_i == iso3_j and year matches
#     mask = (df_full["iso3_i"] == iso3_i) & (df_full["iso3_j"] == iso3_i) & (df_full["year"] == year)
#     df_full.loc[mask, "pip"] = home_pip

In [21]:
# Check home bias for NOR and SWE
last_year_df = df_full[df_full['year'] == df_full['year'].max()]
for country in ['NOR', 'SWE']:
    country_df = last_year_df[last_year_df['iso3_i'] == country]
    home_pip = country_df[country_df['iso3_j'] == country]['pip'].values[0]
    foreign_pip = country_df[country_df['iso3_j'] != country]['pip'].sum()
    calculated_home_bias = home_pip / (home_pip + foreign_pip)
    print(f'{country} - Home Pip: {home_pip:.2f}, Foreign Pip: {foreign_pip:.2f}, Calculated Home Bias: {calculated_home_bias:.2%}')

NOR - Home Pip: nan, Foreign Pip: 4605580.81, Calculated Home Bias: nan%
SWE - Home Pip: nan, Foreign Pip: 1416346.71, Calculated Home Bias: nan%


In [22]:
# Recompute the lagged pip after imputation
df_full = df_full.sort_values(by=["iso3_i", "iso3_j", "year"])
df_full["pip_t-1"] = df_full.groupby(["iso3_i", "iso3_j"])["pip"].shift(1)

In [23]:
df_full.isna().mean().sort_values(ascending=False)

d_cul                       0.953833
MSCI_Return_j               0.798780
market_cap_j                0.703760
pip_t-1                     0.676526
pip                         0.664862
bis_debt_i                  0.587161
TFP_j                       0.573171
MSCI_Return_i               0.568894
market_cap_i                0.538361
Interest Rate_j             0.505589
Alpha_j                     0.502033
Labour Share_j              0.502033
Human Capital_j             0.484248
Worker Population_j         0.366870
Capital Stock_j             0.363313
Depreciation Rate_j         0.363313
TFP_i                       0.353340
Real GDP_j                  0.345528
Total Population_j          0.345528
Investment Share_j          0.345528
language_spoken_share       0.335676
language_distance           0.335676
language_proximity          0.335676
common_official_language    0.335676
Interest Rate_i             0.318633
Alpha_i                     0.316806
Labour Share_i              0.316806
H

In [24]:
# Rename the columns to macroeconomic variables for CMU simulation
COLUMN_MAP = {
    # ── Identifiers ───────────────────────────────────────────
    "iso3_i"                   : "iso3_i",        # destination country (receives capital)
    "iso3_j"                   : "iso3_j",        # origin country (source of capital)
    "year"                     : "year",

    # ── Gravity / bilateral variables (frictions) ─────────────
    "distance_km"              : "d_geo",         # geographic distance (km), will be logged
    "shared_border"            : "border",        # 1 if shared border
    "common_legal_origin"      : "legal_origin",  # common legal origin
    "language_distance"        : "d_ling",        # linguistic distance (will negate for proximity)
    "language_spoken_share"    : "lang_share",    # share of population speaking common language
    "common_official_language" : "lang_official", # 1 if common official language
    # "cultural_distance"        : "d_cul",         # cultural distance (from hofstede)

    # ── CPIS bilateral equity flows (a_ij) ──────────────────
    "pip"                     : "a_ij",          # bilateral equity holdings (USD), will be logged
    "pip_t-1"                 : "a_ij_lag1",     # lagged bilateral holdings

    # ── Returns ───────────────────────────────────────────────
    "MSCI_Return_i"            : "r_i",           # MSCI return destination country i
    "MSCI_Return_j"            : "r_j",           # MSCI return origin country j

    # ── Production side — destination country (i) ─────────────
    "Real GDP_i"               : "Y_i",           # real GDP (output)
    "Capital Stock_i"          : "k_i",           # capital stock for production
    "Worker Population_i"      : "L_i",           # employment (labour)
    "Total Population_i"       : "pop_i",         # total population
    "Labour Share_i"           : "lab_sh_i",      # labour share of income
    "Alpha_i"                  : "alpha_i",       # capital share = 1 - lab_sh_i
    "Depreciation Rate_i"      : "delta_i",       # capital depreciation rate
    "TFP_i"                    : "A_i",           # total factor productivity
    "Interest Rate_i"          : "rf_i",          # risk-free rate
    "Investment Share_i"       : "inv_share_i",   # investment as share of GDP
    "Human Capital_i"          : "hc_i",          # human capital index
    "market_cap_i"             : "M_i",           # stock market capitalization (size of investment opportunities)
    "euro_i"                   : "euro_i",        # 1 if in Eurozone
    
    # ── Production side — origin country (j) ──────────────────
    "Real GDP_j"               : "Y_j",           # real GDP (output)
    "Capital Stock_j"          : "k_j",           # capital stock for production
    "Worker Population_j"      : "L_j",           # employment (labour)
    "Total Population_j"       : "pop_j",         # total population
    "Labour Share_j"           : "lab_sh_j",      # labour share of income
    "Alpha_j"                  : "alpha_j",       # capital share = 1 - lab_sh_j
    "Depreciation Rate_j"      : "delta_j",       # capital depreciation rate
    "TFP_j"                    : "A_j",           # total factor productivity
    "Interest Rate_j"          : "rf_j",          # risk-free rate
    "Investment Share_j"       : "inv_share_j",   # investment as share of GDP
    "Human Capital_j"          : "hc_j",          # human capital index
    "market_cap_j"             : "M_j",           # stock market capitalization (size of investment opportunities)
    "euro_j"                   : "euro_j",        # 1 if in Eurozone
}

# Check which columns do not get renamed
for col in df_full.columns:
    if col not in COLUMN_MAP:
        print(f"Column '{col}' does not have a mapping in COLUMN_MAP and will not be renamed.")

# Apply the renaming
df_full = df_full.rename(columns=COLUMN_MAP)

Column 'd_cul' does not have a mapping in COLUMN_MAP and will not be renamed.
Column 'language_proximity' does not have a mapping in COLUMN_MAP and will not be renamed.
Column 'bis_debt_i' does not have a mapping in COLUMN_MAP and will not be renamed.


In [25]:
# Add the R_i and R_j, which are from the cobb-douglas return formula, which is R = A * (k^alpha) * (L^(1-alpha)) - delta
df_full["R_i"] = df_full["A_i"] * (df_full["k_i"] ** df_full["alpha_i"]) * (df_full["L_i"] ** (1 - df_full["alpha_i"])) - df_full["delta_i"]
df_full["R_j"] = df_full["A_j"] * (df_full["k_j"] ** df_full["alpha_j"]) * (df_full["L_j"] ** (1 - df_full["alpha_j"])) - df_full["delta_j"]

In [26]:
# Apply average to d_cul
# since it updates very rarely
df_full["d_cul"] = df_full.groupby(["iso3_i", "iso3_j"])["d_cul"].transform(lambda x: x.fillna(x.mean()))

In [27]:
remove_cols = ["Index Year_d", "year_d", "year_o", "Year_o", "Year_d"]
df_full = df_full.drop(columns=remove_cols, errors="ignore")

In [28]:
# Fill informational columns that should have a constant values
# Infer them by getting mean std of all columns grouped by
# iso3_i and iso3_j.
grp = df_full.groupby(["iso3_i", "iso3_j"]).agg(["mean", "std", "count"]).reset_index()
display(grp.head())
# Get the columns where mean std is 0, and count > 1 (i.e. there are duplicates with same value)
constant_cols = []
for col in df_full.columns:
    if col in ["iso3_i", "iso3_j", "year"]:
        continue
    mean_std = grp[(col, "std")]
    mean_count = grp[(col, "count")]
    if np.nanmean(mean_std) == 0 and np.nanmean(mean_count) > 1:
        constant_cols.append(col)
print("Constant columns:", constant_cols)

iso3_i iso3_j    year                     a_ij            M_i      ...  \
                   mean      std count      mean std count mean std  ...   
0    ABW    ABW  2020.5  2.44949     8       NaN NaN     0  NaN NaN  ...   
1    ABW    AFG  2020.5  2.44949     8       NaN NaN     0  NaN NaN  ...   
2    ABW    AGO  2020.5  2.44949     8       NaN NaN     0  NaN NaN  ...   
3    ABW    AIA  2020.5  2.44949     8       NaN NaN     0  NaN NaN  ...   
4    ABW    ALB  2020.5  2.44949     8  0.509772 NaN     1  NaN NaN  ...   

  d_ling a_ij_lag1            R_i                     R_j                      
   count      mean std count mean std count          mean           std count  
0      0       NaN NaN     0  NaN NaN     0           NaN           NaN     0  
1      8       NaN NaN     0  NaN NaN     0           NaN           NaN     0  
2      8       NaN NaN     0  NaN NaN     0  40948.409372  22443.494334     7  
3      8       NaN NaN     0  NaN NaN     0           NaN           NaN     0  
4      8  0.509772 NaN     1  NaN NaN     0     36.418947     14.852657     7  

[5 rows x 128 columns]

Constant columns: ['d_geo', 'border', 'legal_origin', 'language_proximity', 'lang_share', 'lang_official', 'euro_i', 'euro_j', 'd_ling']


In [29]:
df_full[constant_cols].isna().mean()

d_geo                 0.060679
border                0.090381
legal_origin          0.092291
language_proximity    0.335676
lang_share            0.335676
lang_official         0.335676
euro_i                0.000000
euro_j                0.000000
d_ling                0.335676
dtype: float64

In [30]:
# Only select the mean values
grp_mean = df_full.groupby(["iso3_i", "iso3_j"])[constant_cols].mean().reset_index()
# Join to df_full, overwrite the existing constant columns
for col in constant_cols:
    del df_full[col]  # Remove the original column with NaNs

df_full = pd.merge(df_full, grp_mean, on=["iso3_i", "iso3_j"], how="left")
df_full[constant_cols].isna().mean()

d_geo                 0.060679
border                0.090381
legal_origin          0.092291
language_proximity    0.335676
lang_share            0.335676
lang_official         0.335676
euro_i                0.000000
euro_j                0.000000
d_ling                0.335676
dtype: float64

In [31]:
# Fill in the bilateral variables for intra-national pairs (where iso3_i == iso3_j)
bil_cols = ["dist", "border", "lang", "lang_share", "lang_official"]

In [32]:
# Drop duplicates, keep last (iso3_i, iso3_j, year)
df_full = df_full.drop_duplicates(subset=["iso3_i", "iso3_j", "year"], keep="last")

In [33]:
# Set all friction columns for the same country to closest
same_fric_vals = {
    "d_geo": 0,
    "border": 1,
    "legal_origin": 1,
    "d_ling": 0,
    "lang_share": 1,
    "lang_official": 1,
}

for col, val in same_fric_vals.items():
    df_full.loc[df_full["iso3_i"] == df_full["iso3_j"], col] = val

In [34]:
# Save to csv
savepath = "../Clean/Final-v4.csv"
df_full.to_csv(savepath, index=False)

In [35]:
df_eu = df_full[(df_full["iso3_i"].isin(COUNTRIES)) & (df_full["iso3_j"].isin(COUNTRIES))]
df_eu[["d_geo"]].isna().mean()

d_geo    0.0
dtype: float64

In [36]:
df_eu.isna().mean().sort_values(ascending=False)

d_cul                 0.765869
M_i                   0.435484
M_j                   0.435484
r_i                   0.419355
r_j                   0.419355
bis_debt_i            0.286290
language_proximity    0.155047
Y_i                   0.125000
lab_sh_i              0.125000
pop_i                 0.125000
k_i                   0.125000
L_i                   0.125000
rf_j                  0.125000
A_j                   0.125000
hc_j                  0.125000
inv_share_j           0.125000
rf_i                  0.125000
inv_share_i           0.125000
delta_i               0.125000
A_i                   0.125000
Y_j                   0.125000
k_j                   0.125000
L_j                   0.125000
pop_j                 0.125000
lab_sh_j              0.125000
delta_j               0.125000
hc_i                  0.125000
alpha_i               0.125000
R_i                   0.125000
R_j                   0.125000
alpha_j               0.125000
lang_share            0.122789
d_ling  

In [37]:
df_eu

,iso3_i,iso3_j,year,a_ij,M_i,M_j,d_cul,r_i,r_j,Y_i,...,R_j,d_geo,border,legal_origin,language_proximity,lang_share,lang_official,euro_i,euro_j,d_ling
38126,AUT,AUT,2017,NaN,1.506460e+11,1.506460e+11,0.0,0.524676,0.524676,5.393304e+05,...,953.693122,0.0,1.0,1.0,NaN,1.0,1.0,1.0,1.0,0.0
38133,AUT,AUT,2018,NaN,1.168020e+11,1.168020e+11,0.0,-0.228792,-0.228792,5.527286e+05,...,978.532863,0.0,1.0,1.0,NaN,1.0,1.0,1.0,1.0,0.0
38140,AUT,AUT,2019,NaN,1.330982e+11,1.330982e+11,0.0,0.170519,0.170519,5.624289e+05,...,923.519795,0.0,1.0,1.0,NaN,1.0,1.0,1.0,1.0,0.0
38147,AUT,AUT,2020,NaN,1.320832e+11,1.320832e+11,0.0,-0.036747,-0.036747,5.268932e+05,...,989.165838,0.0,1.0,1.0,NaN,1.0,1.0,1.0,1.0,0.0
38154,AUT,AUT,2021,NaN,1.628889e+11,1.628889e+11,0.0,0.314996,0.314996,5.521595e+05,...,1079.931701,0.0,1.0,1.0,NaN,1.0,1.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
910811,USA,USA,2020,NaN,4.156986e+13,4.156986e+13,0.0,0.150320,0.150320,2.232918e+07,...,26560.695061,0.0,1.0,1.0,NaN,1.0,1.0,0.0,0.0,0.0
910814,USA,USA,2021,NaN,4.854854e+13,4.854854e+13,0.0,0.255930,0.255930,2.368120e+07,...,32800.433082,0.0,1.0,1.0,NaN,1.0,1.0,0.0,0.0,0.0
910817,USA,USA,2022,NaN,4.029798e+13,4.029798e+13,0.0,-0.171257,-0.171257,2.427613e+07,...,41672.551243,0.0,1.0,1.0,NaN,1.0,1.0,0.0,0.0,0.0
910820,USA,USA,2023,NaN,4.897940e+13,4.897940e+13,0.0,0.177234,0.177234,2.497715e+07,...,47269.574168,0.0,1.0,1.0,NaN,1.0,1.0,0.0,0.0,0.0
